In [1]:
# 🧠 LoRA Fine-Tuning for Industrial Reasoning - Colab Ready
# Copy this entire file into a new Google Colab notebook

# ==============================================================================
# SETUP & INSTALLATION
# ==============================================================================

# Check GPU
!nvidia-smi


Sun Feb 15 18:44:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [6]:
# Install with latest compatible versions
print("Installing dependencies...")
!pip install -q transformers>=4.41.0 peft>=0.10.0 accelerate>=0.27.0 bitsandbytes>=0.43.0
!pip install -q datasets>=2.18.0 trl>=0.8.0

print("✓ Installation complete!")
print("\n⚠️  Click 'Runtime > Restart runtime' now, then run cells below.")


Installing dependencies...
✓ Installation complete!

⚠️  Click 'Runtime > Restart runtime' now, then run cells below.


In [1]:
# ==============================================================================
# AFTER RESTART, RUN FROM HERE
# ==============================================================================

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, DataCollatorForCompletionOnlyLM
from datasets import load_dataset
import json
import random
import os
import re

print("✓ All imports successful!")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")

/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


ImportError: peft>=0.17.0 is required for a normal functioning of this module, but found peft==0.7.1.

In [3]:

import json
import random
import os

print("Generating synthetic industrial reasoning dataset...")

Generating synthetic industrial reasoning dataset...


In [4]:
# Dataset templates
categories = {
    "paint_defects": {
        "locations": ["front bumper", "rear door", "hood", "roof panel", "trunk"],
        "defects": ["peeling", "bubbling", "color mismatch", "orange peel"],
        "contexts": ["high humidity", "after cleaning", "temperature change", "new batch"]
    },
    "assembly_defects": {
        "locations": ["door alignment", "panel gaps", "dashboard", "seat mounting"],
        "defects": ["misalignment", "loose fasteners", "excessive gap"],
        "contexts": ["production speed up", "operator change", "tooling change"]
    },
    "welding_defects": {
        "locations": ["body seams", "floor joints", "pillar welds"],
        "defects": ["porosity", "spatter", "cracks", "incomplete penetration"],
        "contexts": ["power fluctuation", "electrode change", "material batch change"]
    }
}

In [5]:
def generate_sample(category, data):
    """Generate one training sample"""

    if category == "paint_defects":
        loc = random.choice(data["locations"])
        defect = random.choice(data["defects"])
        ctx = random.sample(data["contexts"], 2)

        input_text = f"""{defect.capitalize()} observed on {loc}.

### Context:
- Occurs during {ctx[0]}
- Production line {random.randint(1,3)} only
- Started {ctx[1]}

### Task:
Identify root cause and recommend actions."""

        output_text = f"""### Step-by-step Analysis:

Step 1: Environmental correlation
Issue correlates with {ctx[0]}, suggesting environmental influence on paint application.

Step 2: Production line isolation
Only Line {random.randint(1,3)} affected, indicates localized factor.

Step 3: Temporal analysis
Problem started with {ctx[1]}, points to recent system change.

Step 4: Causal hypothesis
{ctx[0].capitalize()} affects paint curing conditions and coating consistency.

### Root Cause:
Paint process disrupted due to {ctx[0]} combined with recent changes affecting {loc}.

### Recommended Action:
1. Monitor environmental conditions during application
2. Recalibrate paint booth parameters
3. Implement preventive controls for {ctx[0]}
4. Increase quality checks on {loc}"""

    elif category == "assembly_defects":
        loc = random.choice(data["locations"])
        defect = random.choice(data["defects"])
        ctx = random.sample(data["contexts"], 2)

        input_text = f"""{defect.capitalize()} detected in {loc}.

### Context:
- Observed after {ctx[0]}
- Affects {random.randint(10,25)}% of units
- Recent change: {ctx[1]}

### Task:
Identify root cause and recommend actions."""

        output_text = f"""### Step-by-step Analysis:

Step 1: Defect pattern analysis
{defect.capitalize()} occurring in {loc}, affects {random.randint(10,25)}% of units.

Step 2: Recent changes correlation
Coincides with {ctx[0]}, no design changes identified.

Step 3: Process parameter review
Assembly sequence affected by {ctx[1]}.

Step 4: Root cause determination
{ctx[0].capitalize()} introduced variation in assembly process.

### Root Cause:
Assembly variation from {ctx[0]}, resulting in {defect} at {loc}.

### Recommended Action:
1. Conduct assembly process audit
2. Verify tooling and fixture condition
3. Retrain operators on critical steps
4. Implement in-process quality checks
5. Tighten process control parameters"""

    else:  # welding_defects
        loc = random.choice(data["locations"])
        defect = random.choice(data["defects"])
        ctx = random.choice(data["contexts"])

        input_text = f"""{defect.capitalize()} found in {loc}.

### Context:
- Defect: {defect}
- Location: {loc}
- Recent event: {ctx}
- Detection: Ultrasonic testing

### Task:
Identify root cause and recommend actions."""

        output_text = f"""### Step-by-step Analysis:

Step 1: Defect characterization
{defect.capitalize()} detected in {loc} via ultrasonic testing.

Step 2: Process parameter analysis
Recent {ctx} event may have caused parameter deviation.

Step 3: Equipment condition assessment
Welding equipment requires inspection post-{ctx}.

Step 4: Root cause identification
{ctx.capitalize()} caused deviation in welding process.

### Root Cause:
Welding parameter deviation from {ctx}, causing {defect} in {loc}.

### Recommended Action:
1. Recalibrate welding equipment
2. Verify consumables and shielding gas
3. Implement parameter monitoring
4. Conduct 100% inspection until stable
5. Update maintenance procedures"""

    return {
        "instruction": "Analyze the following industrial incident and provide step-by-step reasoning to identify the root cause and recommend corrective actions.",
        "input": input_text,
        "output": output_text,
        "category": category
    }

In [6]:
# Generate dataset
random.seed(42)
samples = []
for _ in range(100):  # 100 samples for quick demo (increase to 800 for real training)
    cat = random.choice(list(categories.keys()))
    samples.append(generate_sample(cat, categories[cat]))


In [7]:
# Shuffle and split
random.shuffle(samples)
train_size = int(len(samples) * 0.7)
val_size = int(len(samples) * 0.15)

train_data = samples[:train_size]
val_data = samples[train_size:train_size + val_size]
test_data = samples[train_size + val_size:]

print(f"✓ Generated {len(samples)} samples")
print(f"  Train: {len(train_data)}, Val: {len(val_data)}, Test: {len(test_data)}")

✓ Generated 100 samples
  Train: 70, Val: 15, Test: 15


In [8]:
# Save to files
os.makedirs('data', exist_ok=True)

with open('data/train.jsonl', 'w') as f:
    for s in train_data:
        f.write(json.dumps(s) + '\n')

with open('data/val.jsonl', 'w') as f:
    for s in val_data:
        f.write(json.dumps(s) + '\n')

with open('data/test.jsonl', 'w') as f:
    for s in test_data:
        f.write(json.dumps(s) + '\n')

print("✓ Dataset saved to data/")

✓ Dataset saved to data/


In [9]:
# Show sample
print("\n=== Sample Input ===")
print(train_data[0]['input'][:200] + "...")
print("\n=== Sample Output ===")
print(train_data[0]['output'][:300] + "...")


=== Sample Input ===
Spatter found in floor joints.

### Context:
- Defect: spatter
- Location: floor joints
- Recent event: material batch change
- Detection: Ultrasonic testing

### Task:
Identify root cause and recomme...

=== Sample Output ===
### Step-by-step Analysis:

Step 1: Defect characterization
Spatter detected in floor joints via ultrasonic testing.

Step 2: Process parameter analysis
Recent material batch change event may have caused parameter deviation.

Step 3: Equipment condition assessment
Welding equipment requires inspecti...


In [14]:
# ==============================================================================
# MODEL LOADING & LORA SETUP
# ==============================================================================
!pip install trl

import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, DataCollatorForCompletionOnlyLM
from datasets import load_dataset

ImportError: cannot import name 'DataCollatorForCompletionOnlyLM' from 'trl' (/usr/local/lib/python3.12/dist-packages/trl/__init__.py)

In [1]:
# ==============================================================================
# MODEL LOADING & LORA SETUP
# ==============================================================================
!pip install trl==0.7.10 transformers==4.37.2 --force-reinstall

import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, DataCollatorForCompletionOnlyLM
from datasets import load_dataset

  Using cached trl-0.7.10-py3-none-any.whl.metadata (10 kB)
  Using cached transformers-4.37.2-py3-none-any.whl.metadata (129 kB)
  Using cached torch-2.10.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (31 kB)
  Using cached numpy-2.4.2-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached accelerate-1.12.0-py3-none-any.whl.metadata (19 kB)
  Using cached datasets-4.5.0-py3-none-any.whl.metadata (19 kB)
  Using cached tyro-1.0.6-py3-none-any.whl.metadata (12 kB)
  Using cached filelock-3.24.0-py3-none-any.whl.metadata (2.0 kB)
  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
  Using cached packaging-26.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached pyyaml-6.0.3-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (2.4 kB)
  Using cached regex-2026.1.15-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (40 kB)
  Using cached requests-2.32.5-p

ImportError: cannot import name 'EncoderDecoderCache' from 'transformers' (/usr/local/lib/python3.12/dist-packages/transformers/__init__.py)

In [3]:
!pip install -q transformers peft accelerate bitsandbytes datasets trl

In [4]:
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments
)

/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


In [5]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, DataCollatorForCompletionOnlyLM
from datasets import load_dataset

ImportError: peft>=0.17.0 is required for a normal functioning of this module, but found peft==0.7.1.

In [ ]:
print("\n" + "="*80)
print("LOADING MODEL")
print("="*80)

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

In [ ]:
# Load model
print("Loading Mistral-7B with 4-bit quantization...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"✓ Model loaded: {MODEL_NAME}")
print(f"  GPU Memory: {model.get_memory_footprint() / 1024**3:.2f} GB")

In [ ]:
# LoRA configuration
lora_config = LoraConfig(
    r=16,                    # Rank
    lora_alpha=32,           # Scaling
    target_modules=[         # Target layers
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

In [ ]:
# Prepare and add LoRA
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)

# Print trainable params
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())

print(f"\n✓ LoRA adapters configured:")
print(f"  Rank: 16, Alpha: 32")
print(f"  Trainable params: {trainable:,} ({100 * trainable / total:.4f}%)")
print(f"  Total params: {total:,}")

In [ ]:
# ==============================================================================
# PREPARE DATASET
# ==============================================================================

print("\n" + "="*80)
print("PREPARING DATASET")
print("="*80)

# Load datasets
train_dataset = load_dataset('json', data_files='data/train.jsonl', split='train')
val_dataset = load_dataset('json', data_files='data/val.jsonl', split='train')

def format_prompt(sample):
    """Format sample for training"""
    return f"""### Instruction:
{sample['instruction']}

### Input:
{sample['input']}

### Response:
{sample['output']}"""

def format_prompts(examples):
    texts = []
    for inst, inp, out in zip(examples['instruction'], examples['input'], examples['output']):
        texts.append(format_prompt({'instruction': inst, 'input': inp, 'output': out}))
    return {'text': texts}

train_dataset = train_dataset.map(format_prompts, batched=True, remove_columns=train_dataset.column_names)
val_dataset = val_dataset.map(format_prompts, batched=True, remove_columns=val_dataset.column_names)

print(f"✓ Training samples: {len(train_dataset)}")
print(f"✓ Validation samples: {len(val_dataset)}")

In [ ]:
# ==============================================================================
# TRAINING CONFIGURATION
# ==============================================================================

print("\n" + "="*80)
print("TRAINING SETUP")
print("="*80)

training_args = TrainingArguments(
    output_dir="./outputs",

    # Training params
    num_train_epochs=3,                    # 3 epochs
    per_device_train_batch_size=4,         # Batch size 4
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=8,          # Effective batch = 32

    # Optimization
    learning_rate=2e-4,
    weight_decay=0.01,
    optim="paged_adamw_8bit",

    # LR schedule
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,

    # Precision
    bf16=True,

    # Logging
    logging_steps=5,
    logging_dir="./logs",
    report_to="none",  # Set to "wandb" if you want W&B logging

    # Evaluation
    eval_strategy="steps",
    eval_steps=20,
    save_steps=20,
    save_total_limit=2,
    load_best_model_at_end=True,

    # Memory
    gradient_checkpointing=True,

    # Misc
    seed=42,
)


In [ ]:
# Data collator (train only on response)
response_template = "\n### Response:\n"
collator = DataCollatorForCompletionOnlyLM(
    response_template=response_template,
    tokenizer=tokenizer
)


In [ ]:
# Initialize trainer
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=collator,
    max_seq_length=2048,
    dataset_text_field="text",
    packing=False,
)

print("✓ Trainer initialized")
print(f"  Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"  Total training steps: ~{len(train_dataset) // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps) * training_args.num_train_epochs}")


In [ ]:
# ==============================================================================
# TRAINING
# ==============================================================================

print("\n" + "="*80)
print("STARTING TRAINING")
print("="*80)
print("This will take 30-60 minutes on T4 GPU for 100 samples...")
print("(For 800 samples, expect 3-4 hours)")
print()

# Train!
trainer.train()

print("\n✓ Training complete!")

# Save final model
trainer.save_model("./final_model")
tokenizer.save_pretrained("./final_model")
print("✓ Model saved to ./final_model")

In [ ]:
# ==============================================================================
# QUICK EVALUATION
# ==============================================================================

print("\n" + "="*80)
print("TESTING MODEL")
print("="*80)

model.eval()

def generate_response(prompt_dict):
    """Generate response from model"""
    full_prompt = format_prompt(prompt_dict)
    full_prompt = full_prompt.split("### Response:")[0] + "### Response:\n"

    inputs = tokenizer(full_prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=400,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response[len(full_prompt):]

In [ ]:
# Test on a sample
test_sample = test_data[0]
print("INPUT:")
print(test_sample['input'])
print("\n" + "-"*80)
print("GENERATED OUTPUT:")
generated = generate_response(test_sample)
print(generated)
print("\n" + "-"*80)
print("GROUND TRUTH:")
print(test_sample['output'][:500] + "...")

In [ ]:
# ==============================================================================
# SIMPLE METRICS
# ==============================================================================

print("\n" + "="*80)
print("QUICK EVALUATION")
print("="*80)

import re

def count_steps(text):
    return len(re.findall(r'Step \d+:', text, re.IGNORECASE))

def has_structure(text):
    required = ["step", "root cause", "action"]
    return sum(1 for term in required if term.lower() in text.lower())

# Evaluate on test set
print("Generating responses for test set...")
responses = []
for sample in test_data[:10]:  # First 10 samples
    resp = generate_response(sample)
    responses.append(resp)

In [ ]:
# Calculate metrics
step_counts = [count_steps(r) for r in responses]
structure_scores = [has_structure(r) for r in responses]
lengths = [len(r.split()) for r in responses]

print(f"\nResults on {len(responses)} test samples:")
print(f"  Avg reasoning steps: {sum(step_counts)/len(step_counts):.1f}")
print(f"  Avg structure score: {sum(structure_scores)/len(structure_scores):.1f}/3")
print(f"  Avg response length: {sum(lengths)/len(lengths):.1f} words")

In [ ]:
# ==============================================================================
# DONE!
# ==============================================================================

print("\n" + "="*80)
print("🎉 TRAINING COMPLETE!")
print("="*80)
print("\nYou have successfully:")
print("✅ Generated synthetic industrial reasoning dataset")
print("✅ Fine-tuned Mistral-7B with LoRA")
print("✅ Achieved structured reasoning outputs")
print("✅ Saved trained model to ./final_model")
print("\nNext steps:")
print("- Scale to 800 samples for full training")
print("- Conduct thorough evaluation")
print("- Compare with base model")
print("- Document results for BMW application")
print("\n🚀 Great work!")